# INSTRUCTOR GUIDE: Week 6 Session 2 Joins + analytical queries at Sunrise Stores
Not for students. Solutions are filled in below each Your turn. Keep `sunrise_stores.db` in the same folder as this notebook.

## Run of show (2 hours)
| Time | Block | What happens |
|---|---|---|
| 0:00 to 0:10 | Check-in | Collect the practice queries if they have them. Ask who already peeked at the Challenge brief. |
| 0:10 to 0:30 | Concept: why we join, join types, grain | Draw customers 1-to-many orders, orders 1-to-many items. The grain check is the whole lesson. |
| 0:30 to 0:55 | Live coding: INNER and LEFT JOIN, count rows before and after | Your turn 1 and 2. Spend time on the 420 vs 1,071 contrast and on the 12 customers with no orders. |
| 0:55 to 1:05 | Break | |
| 1:05 to 1:35 | Analytical queries | Revenue by category, top customers, average order value the careful way. Your turn 3 and 4. |
| 1:35 to 1:50 | Challenge | Monthly revenue, frequent customers, and the grain trap. |
| 1:50 to 2:00 | Recap, Challenge brief, capstone intro | Do not rush the capstone. Even five calm minutes is enough if you point at the brief and the stages. |

**If time runs short:** live-code revenue by category and the LEFT JOIN for unordered products; set average order value as a walk-through in the solution cells; keep 8 to 10 minutes for the capstone brief.

## Talking points
- A join lines up rows that share a key, such as `customer_id` or `order_id`. It is the same idea as `merge` in pandas from Week 4, with different spelling.
- **Grain** means "what does one row represent?" An order is one basket. An order item is one line in that basket. Join them and you get one row per line, not one row per basket. If you then `COUNT(*)` or `AVG` without thinking, you are averaging lines, not orders.
- Always print the row count before and after a join.
- `INNER JOIN` keeps matches only. `LEFT JOIN` keeps every row from the left table and fills gaps with empty values (`NULL`). That is how we find products that were never ordered, and customers who never bought.
- Revenue only counts when `status = 'Completed'`. Cancelled baskets are not sales.
- Average order value must be computed at **order** grain: first add up the lines in each order, then average those totals. Averaging the line amounts directly is the standard trap.
- Capstone: introduced today so they can think about a problem and a dataset. No new method is required tonight. Studio and presentations are in Week 8. Keep this practical and calm; no career talk in the notebook.

## Common mistakes to watch for
- Wrong join key (`products.product_id = orders.order_id` and similar).
- No grain check, then surprise duplicate customers or inflated totals.
- `INNER JOIN` when they meant to find absences (they will never see the 12 silent customers or the two unordered products).
- Forgetting `WHERE o.status = 'Completed'` on money questions.
- `GROUP BY` missing a selected non-aggregate column.
- `SELECT *` after a join, which is noisy and hides duplication.
- For AOV, grouping only by channel on the item table (the wrong-grain query). Contrast it live if you can.

## Expected numbers
- INNER JOIN orders to customers: still **420** rows (one order, one customer).
- JOIN orders to order_items: **1,071** rows (the item grain).
- 12 customers have no orders; 2 products were never ordered (Premium olive oil 1L, Ceramic storage jar).
- Completed revenue by category: Staples 12,210,400 (2,617 units); Beverages 2,265,900 (850); Household 727,600 (462); Snacks 489,900 (459); Personal Care 460,900 (294). Total completed revenue 16,154,700.
- Top spender: customer 64, Lola Eze, Abuja, Wholesale, 20 completed orders, 1,912,700.
- AOV (order grain, completed): WhatsApp 50,278.3 (92 orders); Website 45,630.6 (72); Store 42,493.3 (194). Store still has the most revenue overall (8,243,700) because of volume.
- Peak month: 2026-08, revenue 2,001,000. June is close at 1,993,600.
- 12 customers have more than 8 completed orders (Lola Eze has 20).
- Completed payments: Transfer 159 payments / 6,213,700; Cash 158 / 8,073,100; Card 41 / 1,867,900. Cash is fewer payments than Transfer but more money (larger baskets).


# Week 6 Session 2: Joins and analytical queries at Sunrise Stores
**Forge Academy | Data Science Foundations**

*Covers Week 6 Session 2: joins and analytical queries, the SQL Analytics Challenge, and a first look at the capstone.*

## Mission

Last session we could count, filter and summarise one table at a time. Ngozi's real questions cross tables: "which categories actually brought in money?", "who are the highest-spending customers?", "are any products sitting on the shelf unsold?"

Today we join the tables, we check the grain of the result so we do not double-count, and we answer those questions in SQL. By the end you will have the brief for the SQL Analytics Challenge, and a first look at the capstone that runs through the rest of the course.

Keep `sunrise_stores.db` in the same folder as this notebook and run the cells from top to bottom.


## 0. Reconnect


In [ ]:
import sqlite3


In [ ]:
import pandas as pd


In [ ]:
conn = sqlite3.connect("sunrise_stores.db")


In [ ]:
pd.read_sql("SELECT name FROM sqlite_master WHERE type = 'table' ORDER BY name;", conn)


## 1. Why we join, and what grain means

Each table has a **key**. `customer_id` appears on `customers` and on `orders`. `order_id` appears on `orders`, `order_items` and `payments`. `product_id` appears on `products` and on `order_items`. A **join** uses a key to line the tables up.

**Grain** means: what does one row represent?

- In `orders`, one row is one basket.
- In `order_items`, one row is one line of that basket (for example, two bags of rice).
- One order often has several lines. So if you join `orders` to `order_items`, you no longer have one row per basket. You have one row per line.

That is not a mistake, unless you then treat the result as if it were still one row per basket. The habit is: **count the rows before the join, count them after, and say what a row means now.**


First, the counts we already trust.


In [ ]:
query = """
SELECT 'orders' AS table_name, COUNT(*) AS n_rows FROM orders
UNION ALL
SELECT 'customers', COUNT(*) FROM customers
UNION ALL
SELECT 'order_items', COUNT(*) FROM order_items
UNION ALL
SELECT 'products', COUNT(*) FROM products;
"""


In [ ]:
pd.read_sql(query, conn)


## 2. INNER JOIN: keep the matches

An `INNER JOIN` keeps only rows that find a match on both sides. We start with `orders` and `customers`. Each order has exactly one customer, so the result should still have one row per order: 420 rows.


In [ ]:
query = """
SELECT
    o.order_id,
    o.order_date,
    o.status,
    o.channel,
    c.first_name,
    c.last_name,
    c.city,
    c.segment
FROM orders AS o
INNER JOIN customers AS c
    ON o.customer_id = c.customer_id
LIMIT 8;
"""


In [ ]:
pd.read_sql(query, conn)


The grain check: how many rows does the join produce, compared with 420 orders?


In [ ]:
query = """
SELECT COUNT(*) AS n_rows
FROM orders AS o
INNER JOIN customers AS c
    ON o.customer_id = c.customer_id;
"""


In [ ]:
pd.read_sql(query, conn)


`AS o` and `AS c` are short names for the tables, so `o.order_id` is readable. The count should be 420. We did not inflate anything, because each order matches one customer.

Now join `orders` to `order_items`. Each basket can have several lines, so we should expect more than 420 rows.


In [ ]:
query = """
SELECT COUNT(*) AS n_rows
FROM orders AS o
INNER JOIN order_items AS oi
    ON o.order_id = oi.order_id;
"""


In [ ]:
pd.read_sql(query, conn)


**INSTRUCTOR NOTE.** 1,071 rows, which matches the `order_items` count (every item belongs to an order). Pause here. Ask: "If I now `COUNT(*)` and call it 'number of orders', what have I actually counted?" Answer: lines, not baskets.


That result should be 1,071, the same as the number of order items. The grain is now **one row per line**, not one row per order. If we counted rows here and called it "number of orders", we would be wrong.

`SELECT *` after a join is especially noisy, because every column from both tables appears, and duplicated keys are easy to miss. Name the columns you need.


### Your turn 1

1. Join `order_items` to `products` on `product_id`. Show `item_id`, `product_name`, `category`, `quantity` and `order_items.unit_price` for 8 rows.
2. Count the rows in that join. Does the number match 1,071? What is the grain of the result?


In [ ]:
# SOLUTION
query = """
SELECT
    oi.item_id,
    p.product_name,
    p.category,
    oi.quantity,
    oi.unit_price
FROM order_items AS oi
INNER JOIN products AS p
    ON oi.product_id = p.product_id
LIMIT 8;
"""
pd.read_sql(query, conn)


In [ ]:
# SOLUTION
query = """
SELECT COUNT(*) AS n_rows
FROM order_items AS oi
INNER JOIN products AS p
    ON oi.product_id = p.product_id;
"""
pd.read_sql(query, conn)


**INSTRUCTOR NOTE. Your turn 1.** The count is **1,071**. Grain: one row per order line, now with the product name and category attached. If someone gets a smaller number, they may have used the wrong key. Two products were never ordered, but an INNER JOIN starting from `order_items` will not show that yet; that is the next idea.


## 3. LEFT JOIN: keep the left table, even when there is no match

An `INNER JOIN` hides absences. If a customer never placed an order, they disappear from an inner join of customers to orders. A `LEFT JOIN` keeps every row from the **left** table, and puts empty values (`NULL`) on the right when there is no match.

How many customers have no orders? We start from `customers` and left-join `orders`.


In [ ]:
query = """
SELECT
    c.customer_id,
    c.first_name,
    c.last_name,
    c.city,
    c.segment,
    o.order_id
FROM customers AS c
LEFT JOIN orders AS o
    ON c.customer_id = o.customer_id
WHERE o.order_id IS NULL
ORDER BY c.customer_id;
"""


In [ ]:
no_orders = pd.read_sql(query, conn)


In [ ]:
no_orders


In [ ]:
print(len(no_orders))


`IS NULL` means "this value is empty". Those customers signed up and have not (yet) placed an order. There should be 12 of them, and they are all in the New segment, which makes sense.

The same pattern finds products that were never ordered.


In [ ]:
query = """
SELECT
    p.product_id,
    p.product_name,
    p.category,
    p.unit_price
FROM products AS p
LEFT JOIN order_items AS oi
    ON p.product_id = oi.product_id
WHERE oi.product_id IS NULL
ORDER BY p.product_id;
"""


In [ ]:
pd.read_sql(query, conn)


**INSTRUCTOR NOTE.** 12 customers, all New. Two products: Premium olive oil 1L (Staples, 9,800) and Ceramic storage jar (Household, 6,500). A nice talking point: both are expensive niche items. If a student uses INNER JOIN here they will get zero rows and may think every product sells.


### Your turn 2

Are there any **Completed** orders that have no payment? Start from `orders`, keep only `status = 'Completed'`, left-join `payments`, and keep rows where `payments.payment_id` is empty. Show the count of such orders. (A result of zero is a real answer, if you can say what it means.)


In [ ]:
# SOLUTION
query = """
SELECT COUNT(*) AS n_completed_without_payment
FROM orders AS o
LEFT JOIN payments AS p
    ON o.order_id = p.order_id
WHERE o.status = 'Completed'
  AND p.payment_id IS NULL;
"""
pd.read_sql(query, conn)


**INSTRUCTOR NOTE. Your turn 2.** The count is **0**. Every completed order has a payment. That is worth saying out loud: a zero can be a finding. Cancelled orders have no payment (by design in this database); we filtered them out with `WHERE o.status = 'Completed'`. If someone drops that filter they will see the 41 cancelled orders instead.


## 4. Analytical queries: revenue, with a careful grain

Ngozi's question: **which product categories brought in the most money from completed orders?**

We need three tables: line items (quantity and price), products (category), and orders (status). Revenue for a line is `quantity * unit_price`. We keep only `Completed` orders, then group by category.


In [ ]:
query = """
SELECT
    p.category,
    SUM(oi.quantity) AS units,
    SUM(oi.quantity * oi.unit_price) AS revenue
FROM order_items AS oi
INNER JOIN products AS p
    ON p.product_id = oi.product_id
INNER JOIN orders AS o
    ON o.order_id = oi.order_id
WHERE o.status = 'Completed'
GROUP BY p.category
ORDER BY revenue DESC;
"""


In [ ]:
revenue_by_category = pd.read_sql(query, conn)


In [ ]:
revenue_by_category


Staples should be far ahead, which fits a provisions shop. Cancelled and refunded baskets are excluded, because they are not kept sales.

A second question: **who are the ten customers who have spent the most on completed orders?** This time the grain of the *answer* is one row per customer, even though we pass through orders and items to get there.


In [ ]:
query = """
SELECT
    c.customer_id,
    c.first_name,
    c.last_name,
    c.city,
    c.segment,
    COUNT(DISTINCT o.order_id) AS n_orders,
    SUM(oi.quantity * oi.unit_price) AS spend
FROM customers AS c
INNER JOIN orders AS o
    ON o.customer_id = c.customer_id
INNER JOIN order_items AS oi
    ON oi.order_id = o.order_id
WHERE o.status = 'Completed'
GROUP BY c.customer_id, c.first_name, c.last_name, c.city, c.segment
ORDER BY spend DESC
LIMIT 10;
"""


In [ ]:
pd.read_sql(query, conn)


`COUNT(DISTINCT o.order_id)` is needed because the join to `order_items` would otherwise count lines, not baskets. That is grain awareness in one function.


**INSTRUCTOR NOTE.** Revenue: Staples 12,210,400; Beverages 2,265,900; Household 727,600; Snacks 489,900; Personal Care 460,900. Total 16,154,700. Top ten are all Wholesale; number one is Lola Eze (64), Abuja, 20 orders, 1,912,700. Ask: "Should Ngozi be more excited about Staples, or about wholesale customers?" Both are true, and they are different decisions.


### Your turn 3

Using only completed orders, compare **Staples** and **Snacks**: total units sold and total revenue for each. You can filter with `WHERE p.category IN ('Staples', 'Snacks')`.


In [ ]:
# SOLUTION
query = """
SELECT
    p.category,
    SUM(oi.quantity) AS units,
    SUM(oi.quantity * oi.unit_price) AS revenue
FROM order_items AS oi
INNER JOIN products AS p
    ON p.product_id = oi.product_id
INNER JOIN orders AS o
    ON o.order_id = oi.order_id
WHERE o.status = 'Completed'
  AND p.category IN ('Staples', 'Snacks')
GROUP BY p.category;
"""
pd.read_sql(query, conn)


**INSTRUCTOR NOTE. Your turn 3.** Staples: 2,617 units, 12,210,400. Snacks: 459 units, 489,900. Staples bring in about 25 times the revenue, not because the shop sells nothing else, but because staples are both dearer and much higher volume.


## 5. Average order value, the careful way

"What is the average order value by channel?" sounds simple, and it is the classic place to get the grain wrong.

If you join orders to items and then take `AVG(quantity * unit_price)` grouped by channel, you are averaging **line values**, not basket totals. A four-line order would be treated as four small orders.

The safe pattern is two steps in one query:

1. Build a small inner table with **one row per completed order** and its total value.
2. Average those totals by channel.


In [ ]:
query = """
SELECT
    channel,
    COUNT(*) AS n_orders,
    ROUND(AVG(order_value), 1) AS avg_order_value,
    SUM(order_value) AS revenue
FROM (
    SELECT
        o.order_id,
        o.channel,
        SUM(oi.quantity * oi.unit_price) AS order_value
    FROM orders AS o
    INNER JOIN order_items AS oi
        ON oi.order_id = o.order_id
    WHERE o.status = 'Completed'
    GROUP BY o.order_id, o.channel
)
GROUP BY channel
ORDER BY avg_order_value DESC;
"""


In [ ]:
pd.read_sql(query, conn)


The part in parentheses is a **subquery**: a query that builds a temporary table, here with one row per completed order. Then the outer query averages those rows.

Here is the wrong-grain version, on purpose, so you can see how different the numbers look. This averages line values, not order values.


In [ ]:
query = """
SELECT
    o.channel,
    COUNT(*) AS n_lines,
    ROUND(AVG(oi.quantity * oi.unit_price), 1) AS avg_line_value
FROM orders AS o
INNER JOIN order_items AS oi
    ON oi.order_id = o.order_id
WHERE o.status = 'Completed'
GROUP BY o.channel;
"""


In [ ]:
pd.read_sql(query, conn)


The wrong query has a much smaller "average" and a `COUNT` that is a line count, not an order count. When a number looks like average order value but is only a few thousand naira in a shop that sells 10 kg rice, check the grain.


**INSTRUCTOR NOTE.** Correct AOV: WhatsApp 50,278.3 (92 orders, revenue 4,625,600); Website 45,630.6 (72, 3,285,400); Store 42,493.3 (194, 8,243,700). Store still wins on total revenue because of volume. Wrong-grain averages are around 16,800 to 18,500 with line counts 489 / 178 / 252. Put both results on the screen and ask which one Ngozi should use for "average basket size".


### Your turn 4

For completed orders, show each payment `method` with the number of payments and the total `amount`. Order by total amount, largest first. You will join `payments` to `orders` so you can keep only completed orders.


In [ ]:
# SOLUTION
query = """
SELECT
    p.method,
    COUNT(*) AS n_payments,
    SUM(p.amount) AS total_amount
FROM payments AS p
INNER JOIN orders AS o
    ON o.order_id = p.order_id
WHERE o.status = 'Completed'
GROUP BY p.method
ORDER BY total_amount DESC;
"""
pd.read_sql(query, conn)


**INSTRUCTOR NOTE. Your turn 4.** Cash 158 payments, 8,073,100; Transfer 159 payments, 6,213,700; Card 41 payments, 1,867,900. A nice observation: Transfer is used slightly more often, but Cash carries more money. If they forget the Completed filter, Cash becomes 168 / 8,271,300 and Transfer 163 / 6,371,500 (refunded payments sneak in).


## 6. Challenge (10 to 15 minutes)

1. For completed orders, show **monthly** revenue. Use `strftime('%Y-%m', o.order_date) AS month` to build a year-month label, then group by that label. Show the month, the number of distinct orders, and the revenue, in calendar order. Which month is highest?
2. List customers with **more than 8** completed orders. Show `customer_id`, `first_name`, `last_name` and the order count, highest first.
3. In one or two sentences (a markdown cell is fine): why is the "average line value by channel" query not a safe measure of average order value?


In [ ]:
# SOLUTION — question 1
query = """
SELECT
    strftime('%Y-%m', o.order_date) AS month,
    COUNT(DISTINCT o.order_id) AS n_orders,
    SUM(oi.quantity * oi.unit_price) AS revenue
FROM orders AS o
INNER JOIN order_items AS oi
    ON oi.order_id = o.order_id
WHERE o.status = 'Completed'
GROUP BY strftime('%Y-%m', o.order_date)
ORDER BY month;
"""
pd.read_sql(query, conn)


In [ ]:
# SOLUTION — question 2
query = """
SELECT
    c.customer_id,
    c.first_name,
    c.last_name,
    COUNT(*) AS n_orders
FROM customers AS c
INNER JOIN orders AS o
    ON o.customer_id = c.customer_id
WHERE o.status = 'Completed'
GROUP BY c.customer_id, c.first_name, c.last_name
HAVING COUNT(*) > 8
ORDER BY n_orders DESC;
"""
pd.read_sql(query, conn)


**SOLUTION — question 3.** Average line value treats each item line as if it were a whole order, so a basket with four lines is counted four times, each at the line amount rather than the basket total. Average order value has to be computed at order grain: sum the lines in each order first, then average those sums.


**INSTRUCTOR NOTE. Challenge.**

1. August 2026 is highest at 2,001,000 (35 orders). June is very close at 1,993,600 (34 orders). January 1,746,100; the quietest months in this window are October 1,185,400 and September 1,241,600. If they forget `DISTINCT` on the order count, the `n_orders` column becomes a line count and will look too large, but the revenue can still be right.
2. Twelve customers. Lola Eze (64) has 20; then Ifeoma Chukwu 12; Amaka Chukwu, Bisi Nwosu and Fatima Bello have 11 each; four customers have 10; three have 9. If they use `>= 8` they will pick up more rows. If they forget `Completed` they will include cancelled baskets.


## Recap

- A join lines tables up on a shared key. `INNER JOIN` keeps matches only. `LEFT JOIN` keeps every row from the left table and uses `IS NULL` to find absences.
- Always check the grain: count rows before and after, and say what one row means now.
- Money questions should filter to `status = 'Completed'` unless you have a reason not to.
- `COUNT(DISTINCT order_id)` and "sum first, then average" are two practical ways to stay at the grain you meant.
- A subquery can build a one-row-per-order table before you summarise it.

## The SQL Analytics Challenge

This is the major Week 6 assignment (10% of the course). The full brief is in `Week6_SQL_Analytics_Challenge.txt` in this folder. You will receive a starter notebook and a `queries.sql` template as well.

In short: answer the numbered business questions with SQL, in a file called `queries.sql` with short comments, plus a README that says how to run the queries, plus a results write-up that explains what each answer means. Grain-aware joins and clear `GROUP BY` matter more than clever SQL.

## A first look at the capstone

From this week you can start thinking about the end-of-course capstone (25% of the course). The brief is in the Week 8 folder as `Week8_Capstone_Brief.txt`; a short copy of the idea is also in this week's assignment message.

The capstone is one end-to-end project, using tools you already have (Python, pandas, EDA, charts, statistics, SQL, and later a simple model if it fits). You choose a problem and a dataset. You will not need a new method in Week 8; that week is for finishing, writing, and presenting.

A complete capstone is shaped like this:

Problem, Dataset, Prep, EDA, Stats, Model (if you use one), Evaluation, Insights, Recommendations, Limitations, Future work.

Deliverables in Week 8: a GitHub repository, a notebook that runs from top to bottom, a README, a 7 to 10 minute presentation, and a short written reflection.

This week, a small step is enough: write down one possible question and one possible dataset. You can change your mind later.


## Assignment

**Due:** before our Week 7 Session 1 class. **Submit:** `queries.sql`, a README, and your results / explanation, through the usual submission form.

Follow `Week6_SQL_Analytics_Challenge.txt`. Use `sunrise_stores.db` (SQLite is the expected path for this class). PostgreSQL is not required.

Also, in your README or in a short extra note, write one sentence for a possible capstone question and one sentence for a dataset you might use. This is a starting thought, not a locked decision.

**Rubric (20 points, 4 points each, 10% of the course):** Technical correctness, Problem solving, Analytical reasoning, Documentation, Communication. Full details are in the brief. Join logic and grain awareness come first.

AI tools are allowed, but you must be able to explain every query and every number on our call.
